In [5]:
%pip install --upgrade --quiet  xformers --quiet
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install --upgrade --quiet  python-dotenv --quiet
%pip install accelerate --quiet
%pip install weaviate-client --quiet
%pip install sentence-transformers --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 2.6 MB/s eta 0:00:00


In [2]:
import weaviate

# import weaviate.classes as wvc
from dotenv import load_dotenv, find_dotenv
import os
from langchain.vectorstores.weaviate import Weaviate
from langchain.embeddings.huggingface import HuggingFaceEmbeddings


load_dotenv(find_dotenv("tokens.env"))
WCS_API_KEY = os.getenv("YOUR_WEAVIATE_KEY")
WCS_CLUSTER_URL = os.getenv("YOUR_WEAVIATE_CLUSTER")

client = weaviate.Client(
    url=WCS_CLUSTER_URL,
    auth_client_secret=weaviate.auth.AuthApiKey(WCS_API_KEY),
)
device = "cuda"
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"device": device, "batch_size": 32},
)

# vectorstore = Weaviate(client, index_name="LangChain_0c70358e34034236ba8f84cd318e2c7b",
#                        embedding=embed_model, text_key="text", by_text=False)
# query = "children with benign childhood epilepsy"
# docs = vectorstore.similarity_search_with_score(query)
# print(docs)

c:\Users\luke\anaconda3\envs\MediSearch\Lib\site-packages\weaviate\warnings.py:121: DeprecationWarning: Dep005: You are using weaviate-client version 3.26.2. The latest version is 4.5.0.
            Please consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


In [3]:
client.schema.get()

{'classes': [{'class': 'LangChain_68cba61505b242b09a00347b801a80d8',
   'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2},
    'cleanupIntervalSeconds': 60,
    'stopwords': {'additions': None, 'preset': 'en', 'removals': None}},
   'multiTenancyConfig': {'enabled': False},
   'properties': [{'dataType': ['text'],
     'indexFilterable': True,
     'indexSearchable': True,
     'name': 'text',
     'tokenization': 'word'},
    {'dataType': ['text[]'],
     'description': "This property was generated by Weaviate's auto-schema feature on Fri Feb 23 16:45:14 2024",
     'indexFilterable': True,
     'indexSearchable': True,
     'name': 'authors',
     'tokenization': 'word'},
    {'dataType': ['text'],
     'description': "This property was generated by Weaviate's auto-schema feature on Fri Feb 23 16:45:14 2024",
     'indexFilterable': True,
     'indexSearchable': True,
     'name': 'date',
     'tokenization': 'word'},
    {'dataType': ['text'],
     'description': "This property

In [ ]:
# Try to create Schema manually
class_obj = {
    "class": "LangChain",
    "invertedIndexConfig": {
        "bm25": {"b": 0.75, "k1": 1.2},
        "cleanupIntervalSeconds": 60,
        "stopwords": {"additions": None, "preset": "en", "removals": None},
    },
    "multiTenancyConfig": {"enabled": False},
    "properties": [
        {
            "dataType": ["text"],
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "text",
            "tokenization": "word",
        },
        {
            "dataType": ["text[]"],
            "description": "List of article and possibly book authors",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "authors",
            "tokenization": "word",
        },
        {
            "dataType": ["text"],
            "description": "Publication Date",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "date",
            "tokenization": "word",
        },
        {
            "dataType": ["text"],
            "description": "Journal of Article",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "journal",
            "tokenization": "word",
        },
        {
            "dataType": ["text"],
            "description": "Abstract Text of article without section",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "abstract",
            "tokenization": "word",
        },
        {
            "dataType": ["text"],
            "description": "Article ID used by PubMed",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "pmid_id",
            "tokenization": "word",
        },
        {
            "dataType": ["text"],
            "description": "Article Title",
            "indexFilterable": True,
            "indexSearchable": True,
            "name": "title",
            "tokenization": "word",
        },
    ],
    "replicationConfig": {"factor": 1},
    "shardingConfig": {
        "virtualPerPhysical": 128,
        "desiredCount": 1,
        "actualCount": 1,
        "desiredVirtualCount": 128,
        "actualVirtualCount": 128,
        "key": "_id",
        "strategy": "hash",
        "function": "murmur3",
    },
    "vectorIndexConfig": {
        "skip": False,
        "cleanupIntervalSeconds": 300,
        "maxConnections": 64,
        "efConstruction": 128,
        "ef": -1,
        "dynamicEfMin": 100,
        "dynamicEfMax": 500,
        "dynamicEfFactor": 8,
        "vectorCacheMaxObjects": 1000000000000,
        "flatSearchCutoff": 40000,
        "distance": "cosine",
        "pq": {
            "enabled": False,
            "bitCompression": False,
            "segments": 0,
            "centroids": 256,
            "trainingLimit": 100000,
            "encoder": {"type": "kmeans", "distribution": "log-normal"},
        },
    },
    "vectorIndexType": "hnsw",
    "vectorizer": "text2vec-huggingface",
}

In [7]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from dotenv import load_dotenv
import transformers
from torch import cuda, bfloat16

# load environment variables from .env file
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv("./tokens.env"))

bitsAndBites_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)

model_id = "meta-llama/Llama-2-13b-chat-hf"
hf_auth = os.environ.get("HF_AUTH")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=hf_auth)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bitsAndBites_config,
    device_map="auto",
    do_sample=True,
    token=hf_auth,
)

model.eval()

pipe = pipeline(
    task="text-generation",
    model=model,
    return_full_text=True,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    repetition_penalty=1.1,
)

hf = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:671: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/9.90G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [8]:
from langchain import PromptTemplate, LLMChain

template = """Generate an answer for the following question using only the following contexts as your base of knowledge (dont make up your own answer): {context} Say which of the sources you used. Do not state why you used which source.
Question: {question}
"""  # Working!!!

prompt = PromptTemplate(input_variables=["question", "context"], template=template)

chain = LLMChain(prompt=prompt, llm=hf)

question = "What is electroencephalography?"
context = """Source: Wikipedia: Dogs like to run around and play.
Source: Journal of Medicine: Electroencephalography is a method to measure the weight of a patients feet.
"""

print(chain.predict(question=question, context=context))

Answer: Based on the information from Source: Journal of Medicine, electroencephalography is a method to measure the weight of a patient's feet.


In [12]:
from langchain.chains import RetrievalQA

### your code ###

rag_pipeline = RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    verbose=True,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    chain_type_kwargs={"verbose": True},
)

In [13]:
query = "what are the most common forms of benign epilepsy syndromes for children?"

In [14]:
rag_pipeline(query)



> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes are the most common forms of benign epilepsy syndromes. Although cognitive dysfunctions occur in children with both childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes, the similarity between their patterns of underlying cognitive impairments is not well understood. To describe these patterns, we examined multiple cognitive functions in children with childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes.

The purpose of our study is to describe intellectual functioning in three common childhood epilepsy syndromes - frontal lobe epilepsy (FL

{'query': 'what are the most common forms of benign epilepsy syndromes for children?',
 'result': ' Childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes are the most common forms of benign epilepsy syndromes for children.'}